# training-step-cycle — worked example 1: Fit a two-parameter linear model with the 5-call cycle

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `training-step-cycle`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The canonical PyTorch training step consists of five calls in a fixed order: (1) forward pass to compute predictions, (2) compute the loss, (3) `loss.backward()` to compute gradients, (4) `optimizer.step()` to update parameters, (5) `optimizer.zero_grad()` to clear accumulated gradients before the next iteration. Getting this order wrong — for example, calling `zero_grad` before `backward`, or `step` before `backward` — leads to silent failures or gradient accumulation bugs.

## Worked solution

**Step 1 – Build two leaf tensors.** We create `a = t.tensor([0.5], requires_grad=True)` and `b = t.tensor([0.5], requires_grad=True)`. These are the two parameters we want to fit to `y = 3x + 2`.

**Step 2 – Create the optimizer.** `optimizer = t.optim.SGD([a, b], lr=0.05)`. Both parameters are passed in a list so SGD tracks their gradients together.

**Step 3 – The 5-call loop.** On each iteration:
- `pred = a * x + b` (forward: two-parameter linear model)
- `loss = ((pred - y_target) ** 2).mean()` (MSE loss)
- `losses.append(loss.item())` (snapshot before backward changes the graph)
- `loss.backward()` (backward: computes `a.grad` and `b.grad`)
- `optimizer.step()` (update: nudges `a` and `b` toward the minimum)
- `optimizer.zero_grad()` (clear: resets `a.grad` and `b.grad` to zero)

**Step 4 – Return.** We return the trained parameters and loss trajectory. After enough steps, `a ≈ 3.0` and `b ≈ 2.0`.

In [ ]:
import torch as t

def worked1_two_param_regression(n_steps=60):
    """
    Fit y = ax + b to y = 3x + 2 using SGD.
    Returns (a_final, b_final, losses).
    """
    t.manual_seed(17)
    a = t.tensor([0.5], requires_grad=True)
    b = t.tensor([0.5], requires_grad=True)
    optimizer = t.optim.SGD([a, b], lr=0.05)
    x = t.linspace(-1, 1, 12)
    y_target = 3.0 * x + 2.0

    losses = []
    for _ in range(n_steps):
        pred = a * x + b          # forward
        loss = ((pred - y_target) ** 2).mean()  # loss
        losses.append(loss.item())              # snapshot
        loss.backward()           # backward
        optimizer.step()          # step
        optimizer.zero_grad()     # zero_grad

    return a.detach().clone(), b.detach().clone(), losses

a_fit, b_fit, losses = worked1_two_param_regression()
print(f'a ≈ {a_fit.item():.3f}  (target 3.0)')
print(f'b ≈ {b_fit.item():.3f}  (target 2.0)')
print(f'initial loss: {losses[0]:.4f}, final loss: {losses[-1]:.4f}')